In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv('SGJobData.csv')

# --- STEP 1: Filtering the dataframe to only include rows where the 'status_jobStatus' column is 'Closed' ---
df_cleaned = df[df['status_jobStatus'] == 'Closed'].copy()  #.copy() creates a completely independent clone of the data.

In [14]:
#display(df.describe(include='all'))  # Displaying the summary statistics of the dataframe
#display(df.info())  # Displaying the information about the dataframe, including data types and non-null counts

In [15]:
# --- STEP 2: Dropping unnecessary columns from the dataframe ---
columns_to_drop = [
    "occupationId",
    "metadata_jobPostId",
    "metadata_isPostedOnBehalf",
    "status_id"
]
df_cleaned = df_cleaned.drop(columns=columns_to_drop)

.astype(str): forces everything in the column to be read as a "String" (text). 

.str: Pandas to apply a text-editing command to every single row in this entire column all at once.

.strip(): cuts off any blank spaces at the very beginning or the very end of the text.

In [16]:
# --- STEP 3: Clean Data Types & Whitespace ---
# 1. Fix Date Columns: Convert date columns to datetime format and handle any invalid parsing by setting them as NaT (Not a Time).
date_columns = [
    'metadata_expiryDate',
    'metadata_newPostingDate',
    'metadata_originalPostingDate'
]
for col in date_columns:
    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce')  # errors='coerce' = invalid parsing will be set as NaT

# 2. Clean Text Columns: Convert text columns to string type and remove leading/trailing whitespace.
text_columns = [
    'categories',
    'employmentTypes',
    'positionLevels',
    'postedCompany_name',
    'salary_type',
    'status_jobStatus',
    'title'
]
for col in text_columns:
    df_cleaned[col] = df_cleaned[col].astype(str).str.strip()  # Convert all columns to string and remove leading/trailing whitespace


In [17]:
df_cleaned.info()  # Displaying the information about the cleaned dataframe, including data types and non-null counts

<class 'pandas.core.frame.DataFrame'>
Int64Index: 119701 entries, 0 to 1048583
Data columns (total 18 columns):
 #   Column                              Non-Null Count   Dtype         
---  ------                              --------------   -----         
 0   categories                          119701 non-null  object        
 1   employmentTypes                     119701 non-null  object        
 2   metadata_expiryDate                 119701 non-null  datetime64[ns]
 3   metadata_newPostingDate             119701 non-null  datetime64[ns]
 4   metadata_originalPostingDate        119701 non-null  datetime64[ns]
 5   metadata_repostCount                119701 non-null  int64         
 6   metadata_totalNumberJobApplication  119701 non-null  int64         
 7   metadata_totalNumberOfView          119701 non-null  int64         
 8   minimumYearsExperience              119701 non-null  int64         
 9   numberOfVacancies                   119701 non-null  int64         
 10  positio

In [18]:
# --- STEP 4: Duplicates and Missing Values ---
# 1. Drop duplicates based ONLY on these 5 columns matching
df_cleaned = df_cleaned.drop_duplicates(subset=[
    'postedCompany_name',
    'title',
    'categories',
    'positionLevels',
    'salary_minimum',
])  # Remove duplicate rows from the dataframe

# 2. Drop rows where the salary is physically missing.
df_cleaned = df_cleaned.dropna(subset=['salary_minimum', 'salary_maximum'])  

# 3. Handle missing text data by filling with 'Unknown' or a similar placeholder.
df_cleaned[text_columns] = df_cleaned[text_columns].replace(['', 'nan'], 'Unknown') # Replace empty strings and 'nan' with 'Unknown' in text columns

#print("Remaining missing values per column:")
#missing_check = df_cleaned.isna().sum()
#print(missing_check[missing_check > 0])
print(f"\nTotal rows ready for outlier filtering: {len(df_cleaned)}")


Total rows ready for outlier filtering: 95878


In [19]:
# --- STEP 5: Salary Outliers and Logic Checks ---
# 1. Swap salary_minimum and salary_maximum if they are in the wrong order.
swapped_mask = df_cleaned['salary_minimum'] > df_cleaned['salary_maximum']
df_cleaned.loc[swapped_mask, ['salary_minimum', 'salary_maximum']] = df_cleaned.loc[swapped_mask, ['salary_maximum', 'salary_minimum']].values

# 2. Logical market salary range check: Remove rows where the salary is outside a reasonable range (e.g., less than $1_000 or greater than $100,000).
salary_range_mask = (df_cleaned['salary_minimum'] >= 1000) & (df_cleaned['salary_maximum'] <= 100000)
df_cleaned = df_cleaned[salary_range_mask]  # Keep only rows where the salary is within the reasonable range

# 3. Recalculate Average Salary
df_cleaned['salary_average'] = (df_cleaned['salary_minimum'] + df_cleaned['salary_maximum']) / 2

print(f"Data cleaning complete! Final row count: {len(df_cleaned)}\n")
print(df_cleaned[['salary_minimum', 'salary_maximum', 'salary_average']].describe())

Data cleaning complete! Final row count: 94213

       salary_minimum  salary_maximum  salary_average
count    94213.000000    94213.000000    94213.000000
mean      3944.403405     5757.886534     4851.144969
std       2499.431844     4025.309545     3199.461944
min       1000.000000     1000.000000     1000.000000
25%       2500.000000     3300.000000     2900.000000
50%       3000.000000     4500.000000     3800.000000
75%       4500.000000     7000.000000     5750.000000
max      80000.000000   100000.000000    90000.000000


The try/except block: If any row has a missing value or a typo json.loads will throw a fatal error and stop the entire script. The except block catches the error, outputs an empty dictionary {}, and keeps the code running.

In [20]:
# --- STEP 6: Exploding the Categories Column with json formatting ---
import json

# 1. Create a function to extract ALL categories from the JSON array

def extract_all_categories(raw_text):
    try:
        parsed_list = json.loads(raw_text)
        # Loop through the list and grab the 'category' text from every dictionary
        return [item['category'] for item in parsed_list if 'category' in item]
    except:
        # If it's broken or blank, return an empty list
        return []

# 2. Overwrite the column with our clean list of categories
df_cleaned['categories'] = df_cleaned['categories'].apply(extract_all_categories)

# 3. THE MAGIC STEP: Explode the lists into their own rows!
df_cleaned = df_cleaned.explode('categories')

# 4. (Optional) Strip whitespace again just in case the JSON had messy spaces
df_cleaned['categories'] = df_cleaned['categories'].astype(str).str.strip()

# View the results to see how the rows duplicated perfectly
print(f"New row count after explosion: {len(df_cleaned)}")


New row count after explosion: 149971


In [ ]:
def p25(x):
    return x.quantile(0.25)

# 1. Grouping by all five columns
sme_benchmarks = df_cleaned.groupby([
    'categories', 
    'title',              
    'positionLevels', 
    'minimumYearsExperience',
    'postedCompany_name'
])['salary_minimum'].agg(
    Entry_Salary_25th=p25,
    Median_Salary='median',
    Total_Postings='count'
).reset_index()

sme_benchmarks.to_csv('sme_salary_benchmarks.csv', index=False)

In [ ]:
# If you also want to save the raw, cleaned dataset for future deep-dives:
df_cleaned.to_csv('cleaned_raw_job_data.csv', index=False)

print("Export complete! Files are ready for your dashboard.")

Export complete! Files are ready for your dashboard.
